# Metacatalog reliability tiers

Build **cleaned** (soft exclude) and **gold** (hard include) subsets from an existing
metacatalog + LST/sources tree under `CATALOG_DIR`.

**Semantics (grill-me locked):**
- Soft exclude removes only on *positive finite* failures; missing evidence keeps the row.
- Hard include requires every gate to be *calculable* (non-NaN) and pass.
- Multi-image: `n_lst ≥ 2` **or** ≥2 bands with unique association (`n_assoc==1`; Full counts).
- Residuals: absolute Jy/beam on the **origin-band seeded LST row** only (dual RMS/mean thresh, default 1.0).
- Jitter: seed-band rematch members; fail if RMS `> 0.3 × BMAJ`.
- α corner cuts are **not** used for selection.
- Both products are first-class; contract `gold ⊆ cleaned`.

Requires `lwa-catalog[analyze]` (`healpy` for binning, `lwa-healpix` for HiPS export).

Set ``REUSE_CACHED_RELIABILITY = True`` (default) to load existing
``metacatalog_cleaned.parquet`` / ``metacatalog_gold.parquet`` instead of re-filtering.

**Run cells in order.**


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lwa_catalog.analyze import (
    ReliabilityConfig,
    ReliabilityResult,
    filter_metacatalog_reliability,
    metacatalog_to_healpix,
    write_healpix_hips,
)
from lwa_catalog.io import read_all_lst_merged, read_metacatalog, read_table, write_table
from lwa_catalog.paths import CatalogLayout

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coadd2")  # existing fusion tree
NSIDE = 256
REUSE_CACHED_RELIABILITY = True  # load cleaned/gold Parquet if present; set False to regenerate
# HiPS output directory name under CATALOG_DIR; fields: {name} (full|cleaned|gold), {nside}
HIPS_DIR_TEMPLATE = "metacatalog_coadd2_{name}.hips"
CONFIG = ReliabilityConfig(
    resid_rms_thresh_jy=1.0,
    resid_mean_thresh_jy=1.0,
    jitter_bmaj_frac=0.3,
    strict=False,  # True → raise if gold ⊈ cleaned
)

layout = CatalogLayout(CATALOG_DIR)
PATHS = {
    "cleaned": layout.root / "metacatalog_cleaned.parquet",
    "gold": layout.root / "metacatalog_gold.parquet",
    "cleaned_flags": layout.root / "metacatalog_cleaned_flags.parquet",
    "gold_flags": layout.root / "metacatalog_gold_flags.parquet",
}
print("CATALOG_DIR =", layout.root.resolve())
print("REUSE_CACHED_RELIABILITY =", REUSE_CACHED_RELIABILITY)
print("HIPS_DIR_TEMPLATE =", HIPS_DIR_TEMPLATE)


CATALOG_DIR = /fast/claw/metacatalog_coadd2
REUSE_CACHED_RELIABILITY = True
HIPS_DIR_TEMPLATE = metacatalog_coadd2_{name}.hips


## Load metacatalog + LST-merged cache

In [2]:
metacatalog = read_metacatalog(layout)
print(f"metacatalog rows: {len(metacatalog)}")

_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    lst_merged = None
    print("Cache hit for cleaned/gold — skipping LST-merged load")
else:
    lst_merged = read_all_lst_merged(layout)
    for band, df in lst_merged.items():
        print(f"  LST {band}: {len(df)} rows")


metacatalog rows: 28332
  LST Full: 15452 rows
  LST Blue: 28976 rows
  LST Green: 15797 rows
  LST Red: 6268 rows


## Load or run cleaned + gold filters

With ``REUSE_CACHED_RELIABILITY=True`` (default), load ``metacatalog_cleaned.parquet`` /
``metacatalog_gold.parquet`` (+ flags if present). Set the flag to ``False`` to regenerate.


In [3]:
def _reliability_from_parquet(catalog_path: Path, flags_path: Path) -> ReliabilityResult:
    cat = read_table(catalog_path)
    flags = read_table(flags_path) if flags_path.is_file() else pd.DataFrame()
    if "meta_id" in cat.columns:
        meta_ids = cat["meta_id"].to_numpy(dtype=int)
    else:
        meta_ids = np.arange(len(cat), dtype=int)
    return ReliabilityResult(
        catalog=cat,
        meta_ids=meta_ids,
        tier_counts=pd.DataFrame(columns=["tier", "n_in", "n_out", "n_removed"]),
        flags=flags,
        warnings=[f"loaded from {catalog_path.name}"],
    )


_cache_ready = (
    REUSE_CACHED_RELIABILITY
    and PATHS["cleaned"].is_file()
    and PATHS["gold"].is_file()
)
if _cache_ready:
    cleaned = _reliability_from_parquet(PATHS["cleaned"], PATHS["cleaned_flags"])
    gold = _reliability_from_parquet(PATHS["gold"], PATHS["gold_flags"])
    print(f"Loaded cached cleaned={len(cleaned.catalog)}  gold={len(gold.catalog)}")
else:
    if REUSE_CACHED_RELIABILITY:
        print("Cache missing — regenerating cleaned/gold…")
    cleaned, gold = filter_metacatalog_reliability(
        metacatalog,
        layout,
        config=CONFIG,
        lst_merged=lst_merged,
    )

print("=== cleaned (soft exclude) ===")
display(cleaned.tier_counts)
print(f"n_cleaned = {len(cleaned.catalog)}")

print("\n=== gold (hard include) ===")
display(gold.tier_counts)
print(f"n_gold = {len(gold.catalog)}")

assert set(gold.meta_ids).issubset(set(cleaned.meta_ids)), "gold ⊆ cleaned violated"
print("nesting OK: gold ⊆ cleaned")

display(cleaned.catalog.head(5))
display(gold.catalog.head(5))


Cache missing — regenerating cleaned/gold…
=== cleaned (soft exclude) ===


,tier,n_in,n_out,n_removed
0,E0,28332,28332,0
1,E1,28332,25014,3318
2,E2,25014,24800,214
3,E3,24800,24276,524
4,E4,24276,23897,379


n_cleaned = 23897

=== gold (hard include) ===


,tier,n_in,n_out,n_removed
0,I0,28332,28332,0
1,I1,28332,25014,3318
2,I2,25014,24800,214
3,I3,24800,24276,524
4,I4,24276,23897,379
5,I5,23897,22673,1224


n_gold = 22673
nesting OK: gold ⊆ cleaned


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,PA_Red,DC_Maj_Red,DC_Min_Red,DC_PA_Red,source_file_Red,alpha_RG,E_alpha_RG,alpha_GB,E_alpha_GB,meta_id
3,Full,"Full,Blue,Green,Red",6.348186,64.146156,163.019512,171.570647,0.240356,0.190673,46.378977,0.061352,...,40.651416,0.136991,0.118176,168.375277,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-3.072199,0.017692,1.415644,0.015790,3
4,Full,"Full,Blue,Green,Red",49.954183,41.505785,135.316533,244.425446,0.302318,0.234135,45.961461,0.194975,...,43.726367,0.092338,0.079721,73.044957,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-1.486371,0.010918,-3.023406,0.011901,4
11,Full,"Full,Blue,Green,Red",303.616292,23.594900,92.506421,143.375336,0.281266,0.220606,47.760956,0.154447,...,44.037417,0.100318,0.000000,124.168018,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.523062,0.024026,-0.985912,0.013710,11
31,Full,"Full,Blue,Green,Red",176.345686,31.566925,59.511172,66.167454,0.239072,0.181153,42.531192,0.072471,...,39.984455,0.084772,0.052146,112.471691,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.691105,0.048718,-2.554232,0.037478,30
32,Full,"Full,Blue,Green,Red",184.841707,5.829223,58.630997,70.279730,0.255831,0.212284,53.719810,0.104656,...,48.420428,0.104994,0.060644,88.332026,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.311506,0.048829,-0.670350,0.030191,31


,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,DC_Maj,...,PA_Red,DC_Maj_Red,DC_Min_Red,DC_PA_Red,source_file_Red,alpha_RG,E_alpha_RG,alpha_GB,E_alpha_GB,meta_id
31,Full,"Full,Blue,Green,Red",176.345686,31.566925,59.511172,66.167454,0.239072,0.181153,42.531192,0.072471,...,39.984455,0.084772,0.052146,112.471691,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.691105,0.048718,-2.554232,0.037478,30
32,Full,"Full,Blue,Green,Red",184.841707,5.829223,58.630997,70.279730,0.255831,0.212284,53.719810,0.104656,...,48.420428,0.104994,0.060644,88.332026,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.311506,0.048829,-0.670350,0.030191,31
45,Full,"Full,Blue,Green,Red",279.608123,17.191514,44.401801,64.044441,0.270970,0.217274,40.803074,0.134521,...,41.838555,0.135174,0.054686,127.249753,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-0.870262,0.102038,-0.850172,0.065763,44
46,Full,"Full,Blue,Green,Red",101.352590,21.360960,43.982206,47.187289,0.225933,0.182092,39.818069,0.060041,...,43.274159,0.147532,0.091406,56.040075,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-1.215796,0.063032,-1.192387,0.041418,45
52,Full,"Full,Blue,Green,Red",240.635935,1.961917,42.853938,41.646921,0.228244,0.197046,49.371563,0.000000,...,43.750336,0.160568,0.145281,47.149615,Red_I_deep_Taper_Robust+0.0_dewarped_aligned_c...,-1.008566,0.052347,-1.289061,0.035700,51


## Write subsets + companion flags (never overwrite `metacatalog.parquet`)

In [4]:
out = layout.root
paths = PATHS
assert paths["cleaned"].name != "metacatalog.parquet"

if _cache_ready:
    print("Skipping write — using cached cleaned/gold Parquet")
    for label, p in paths.items():
        status = "ok" if p.is_file() else "missing"
        print(f"  {label}: {p} ({status})")
else:
    write_table(cleaned.catalog, paths["cleaned"])
    write_table(gold.catalog, paths["gold"])
    write_table(cleaned.flags, paths["cleaned_flags"])
    write_table(gold.flags, paths["gold_flags"])
    for label, p in paths.items():
        print(f"{label}: {p} ({p.stat().st_size} bytes)")


cleaned: /fast/claw/metacatalog_coadd2/metacatalog_cleaned.parquet (7717164 bytes)
gold: /fast/claw/metacatalog_coadd2/metacatalog_gold.parquet (7214245 bytes)
cleaned_flags: /fast/claw/metacatalog_coadd2/metacatalog_cleaned_flags.parquet (1178969 bytes)
gold_flags: /fast/claw/metacatalog_coadd2/metacatalog_gold_flags.parquet (1178969 bytes)


## Peak_flux-weighted HEALPix → HiPS

Bins each catalog to an equatorial HEALPix map (`nside=NSIDE`), then writes a
**HiPS** tile set with `lwa_healpix.healpix_to_hips` (not FITS).

Install: `pip install 'lwa-catalog[analyze]'` (`healpy` + `lwa-healpix`).

View with Aladin Lite: serve the output directory over HTTP and open `index.html`.


In [5]:
# Peak_flux-weighted HEALPix → HiPS (via lwa-healpix)
# Install: pip install 'lwa-catalog[analyze]'  (healpy + lwa-healpix)

out = layout.root
hips_dirs = {}
for name, cat in (
    ("full", metacatalog),
    ("cleaned", cleaned.catalog),
    ("gold", gold.catalog),
):
    m = metacatalog_to_healpix(cat, nside=NSIDE, weight_col="Peak_flux")
    hips_dir = out / HIPS_DIR_TEMPLATE.format(name=name, nside=NSIDE)
    hips_dirs[name] = write_healpix_hips(
        m,
        hips_dir,
        nest=False,
        coord_frame="equatorial",
        threads=True,
        properties={"obs_title": f"metacatalog {name} (Peak_flux weight)"},
    )
    print(f"{name}: sum={float(m.sum()):.4g} → {hips_dirs[name]}")

print(
    "\nServe a HiPS directory over HTTP and open index.html, e.g.:\n"
    f"  python -m http.server 8000 --directory {hips_dirs['gold']}\n"
    "  then browse http://localhost:8000/"
)


full: sum=1.067e+05 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_full.hips
cleaned: sum=8.789e+04 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_cleaned.hips
gold: sum=8.091e+04 → /fast/claw/metacatalog_coadd2/metacatalog_coadd2_gold.hips

Serve a HiPS directory over HTTP and open index.html, e.g.:
  python -m http.server 8000 --directory /fast/claw/metacatalog_coadd2/metacatalog_coadd2_gold.hips
  then browse http://localhost:8000/
